In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Chargement des documents
# On utilise DirectoryLoader pour charger tous les fichiers .txt du dossier
path_data = "data/hospitals_complete"

# On force l'encodage utf-8 pour bien lire les accents (Hôpital, Médecine, etc.)
loader = DirectoryLoader(
    path_data, 
    glob="**/*.txt", 
    loader_cls=TextLoader, 
    loader_kwargs={'encoding': 'utf-8'}
)

raw_documents = loader.load()

print(f"Nombre de fichiers chargés : {len(raw_documents)}")
# Affiche un aperçu du premier fichier pour vérifier
print(f"Exemple contenu : \n{raw_documents[0].page_content[:200]}")


Nombre de fichiers chargés : 831
Exemple contenu : 
HÔPITAL: Mohamed V

--- INFORMATIONS GÉNÉRALES ---
ID: 1
T


In [2]:
# 2. Découpage du texte (Chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Assez grand pour contenir un bloc "Services" ou "Equipements" entier
    chunk_overlap=200 # Chevauchement pour ne pas couper une phrase importante
)

documents = text_splitter.split_documents(raw_documents)

print(f"Nombre de chunks générés : {len(documents)}")

Nombre de chunks générés : 3166


In [3]:
# 3. Création des Embeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. Création de la base vectorielle FAISS
vectorstore = FAISS.from_documents(documents, embedding)

# 5. Sauvegarde locale (optionnel, pour ne pas tout refaire à chaque fois)
vectorstore.save_local("faiss_index_hospitals")

print("Indexation terminée et sauvegardée.")


Indexation terminée et sauvegardée.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

texts = text_splitter.split_documents([doc])


NameError: name 'doc' is not defined

In [5]:
import os
from operator import itemgetter

# --- TES IMPORTS AUTORISÉS (Pas de langchain.memory) ---
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# --- 1. CHARGEMENT ROBUSTE DU .ENV ---
# On vérifie où on est pour être sûr de trouver le fichier
print(f"📍 Dossier actuel : {os.getcwd()}")

api_key = None
env_filename = ".env"

# Si le fichier n'est pas trouvé direct, on cherche un niveau au-dessus (au cas où)
if not os.path.exists(env_filename):
    if os.path.exists(f"../{env_filename}"):
        env_filename = f"../{env_filename}"
        print(f"⚠️ .env trouvé dans le dossier parent : {env_filename}")

if os.path.exists(env_filename):
    with open(env_filename, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("api="):
                api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                print("✅ Clé API trouvée et chargée.")
                break
else:
    print(f"❌ ERREUR : Le fichier {env_filename} est introuvable au chemin actuel.")

if not api_key:
    raise ValueError("Impossible de continuer sans clé API.")

# --- 2. CONFIGURATION LLM ---
llm = ChatGroq(
    model_name="llama3-70b-8192",
    temperature=0,
    api_key=api_key
)

# --- 3. CHARGEMENT FAISS ---
# Ton dossier s'appelle exactement "faiss_index_hospitals" sur l'image
index_path = "faiss_index_hospitals"

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

try:
    vectorstore = FAISS.load_local(
        index_path, 
        embedding, 
        allow_dangerous_deserialization=True
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    print(f"✅ Index FAISS chargé depuis '{index_path}'")
except Exception as e:
    print(f"❌ Erreur chargement FAISS : {e}")
    print(f"Vérifie que le dossier '{index_path}' contient bien index.faiss et index.pkl")
    raise e

# --- 4. LA CHAÎNE (LCEL - Sans Memory Module) ---
template_text = """
Tu es un assistant expert médical.
Utilise l'historique et le contexte pour répondre.

Historique :
{chat_history}

Contexte trouvé dans la base de données :
{context}

Question de l'utilisateur :
{question}

Réponse :
"""
prompt = PromptTemplate.from_template(template_text)

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# La chaîne de traitement
rag_chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# --- 5. FONCTION D'INTERROGATION ---
# Gestion manuelle de la mémoire (String buffer)
chat_history_buffer = ""

def ask_bot(question: str):
    global chat_history_buffer
    
    print(f"\n🤖 Réflexion en cours sur : '{question}'...")
    
    try:
        response = rag_chain.invoke({
            "question": question, 
            "chat_history": chat_history_buffer
        })
        
        # On ajoute l'échange à l'historique pour la prochaine fois
        chat_history_buffer += f"\nHuman: {question}\nAI: {response}"
        
        print(f"💡 Réponse : {response}")
        return response
    except Exception as e:
        print(f"Erreur lors de l'appel API : {e}")

# --- TEST ---
# ask_bot("Quels sont les hôpitaux disponibles ?")

📍 Dossier actuel : c:\Users\ilyas\Desktop\rag_git
✅ Clé API trouvée et chargée.
✅ Index FAISS chargé depuis 'faiss_index_hospitals'


In [6]:
# On utilise le nouveau standard de Groq : "llama-3.3-70b-versatile"

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",  # <--- CHANGEMENT ICI
    temperature=0,
    api_key=api_key # On réutilise la clé chargée précédemment
)

# --- ON DOIT RECONSTRUIRE LA CHAÎNE ---
# (Car l'ancienne chaîne contient encore l'ancien LLM buggé)

rag_chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt
    | llm  # Le nouveau LLM
    | StrOutputParser()
)

print("✅ Modèle mis à jour vers Llama 3.3 et chaîne reconstruite.")

# --- TU PEUX RÉESSAYER TES QUESTIONS ---
ask_bot("Quels sont les services disponibles à l'hôpital Mohamed V ?")

✅ Modèle mis à jour vers Llama 3.3 et chaîne reconstruite.

🤖 Réflexion en cours sur : 'Quels sont les services disponibles à l'hôpital Mohamed V ?'...
💡 Réponse : Il existe plusieurs hôpitaux Mohamed V dans la base de données, chacun avec des services médicaux différents. Voici les services disponibles pour chaque hôpital Mohamed V :

1. Hôpital Mohamed V (ID : 112) :
 * Urgences 24/7
 * Consultation Générale
 * Cardiologie
 * Pédiatrie
 * Gynécologie Obstétrique

2. Hôpital Mohamed V (ID : 12) :
 * Urgences 24/7
 * Consultation Générale
 * Cardiologie
 * Pédiatrie

Il est important de noter que les services médicaux peuvent varier en fonction de l'hôpital spécifique et de sa localisation. Il est recommandé de contacter directement l'hôpital pour obtenir des informations à jour et précises sur les services disponibles.


"Il existe plusieurs hôpitaux Mohamed V dans la base de données, chacun avec des services médicaux différents. Voici les services disponibles pour chaque hôpital Mohamed V :\n\n1. Hôpital Mohamed V (ID : 112) :\n * Urgences 24/7\n * Consultation Générale\n * Cardiologie\n * Pédiatrie\n * Gynécologie Obstétrique\n\n2. Hôpital Mohamed V (ID : 12) :\n * Urgences 24/7\n * Consultation Générale\n * Cardiologie\n * Pédiatrie\n\nIl est important de noter que les services médicaux peuvent varier en fonction de l'hôpital spécifique et de sa localisation. Il est recommandé de contacter directement l'hôpital pour obtenir des informations à jour et précises sur les services disponibles."